# 5. The real data (free or WRDS)

## What you will learn

- The two ways to build the model's real table: **free public data** (anyone) and **WRDS** (the lecture's
  licensed route).
- How to rebuild every file with one command, and how to read what it writes.
- How many annual reports and downgrades the real table has, what the models scored, and how many companies
  the flag line caught.

## What you need

- Notebooks 1 to 3 help, but you can start here if you know what the panel, the label, the models and the flag line are.
- Python with the packages in `model/requirements.txt` and `jupyter`.
- **Free route (default).** No account. It downloads public files: the SEC ticker list and each company's SEC facts, the rating histories, the
  Loughran-McDonald files, FRED, the Philadelphia Fed files and Yahoo prices. Free prices often leave out companies that
  stopped trading, so the free route usually covers only **about half** the companies in the panel. Its results are
  therefore not the lecture's results.
- **WRDS route.** The lecture's route. It needs **your own WRDS account** with CRSP and Compustat. On the lecture's saved
  data the package's code rebuilds the lecture's table and numbers exactly; the WRDS download step itself has not yet
  been run against WRDS. Everything it writes holds licensed data: keep it on your own computer.
- **Disk and time (rough).** The free route needs **several GB** on disk. On 13 November 2026 it took about 30 minutes
  on an 8-core cloud machine; a home connection may be slower. The commands print the size of each file and the
  seconds for each step; the sizes change every month. The largest parts are the Loughran-McDonald files (the
  10-K header file about 540 MB and the 10-X summaries about 188 MB), the company-facts files and the Yahoo price
  files (about 250 MB in total). The WRDS route
  downloads only the linked stocks, so it uses less disk and usually less time, but it depends on your library's
  connection.
- The notebook does not delete your real data at the end; it is your work.

> **The real data are not in this package.** You make them yourself, under each source's terms. The cells below read
> the files only; they do not contain any real result. Every number you see comes from the files in `OUT_DIR`.

## Set up

The first cell only sets the folders. `DATA_DIR` is where downloads and hand-downloaded files live; `OUT_DIR` is where
the pipeline writes its tables and results. Both can be changed. The notebook also reads the two environment variables
`TCF_NB_DATA_DIR` and `TCF_NB_OUT_DIR` if they are set (the automatic test uses them), so it can run on a tiny example
folder.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd

# Find the repository folder (the one that holds model/src/tcf_credit). This works from the repository folder or from
# model/notebooks.
here = Path.cwd().resolve()
root = next(p for p in [here, *here.parents] if (p / "model" / "src" / "tcf_credit").is_dir())
SRC = root / "model" / "src"
sys.path[:0] = [str(SRC)]
old_path = os.environ.get("PYTHONPATH")
os.environ["PYTHONPATH"] = os.pathsep.join([str(SRC)] + ([old_path] if old_path else []))

DATA_DIR = Path(os.environ.get("TCF_NB_DATA_DIR", root / "data"))
OUT_DIR = Path(os.environ.get("TCF_NB_OUT_DIR", root / "build"))
ROUTE = "free"                                        # change to "wrds" for the lecture's route

pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 40)
print("repository:", root)
print("data folder:", DATA_DIR)
print("output folder:", OUT_DIR)
print("route:", ROUTE)

## Build the outputs, or reuse them

The next cell runs `python -m tcf_credit.rebuild`. That command runs every pipeline step in order, prints one line per
step, skips a step whose output files already exist, and stops at the first failure. If `OUT_DIR/run/pooled.csv` already
exists, the notebook reads the results instead of rebuilding. If the rebuild stops (most often because a file must
be downloaded by hand), the cells below say so and wait; nothing breaks. Delete `OUT_DIR` (or run with `--from 1`) when you want a
fresh build.

In [ ]:
pooled_path = OUT_DIR / "run" / "pooled.csv"
if pooled_path.exists():
    print("Found", pooled_path)
    print("Reading the existing results; no rebuild.")
else:
    command = [sys.executable, "-m", "tcf_credit.rebuild", str(DATA_DIR), str(OUT_DIR), "--route", ROUTE]
    print("Running:", " ".join(command), "\n")
    result = subprocess.run(command)
    if result.returncode != 0:
        print("\nThe rebuild stopped (see its message above). Usually a file must be downloaded by hand first:")
        print("save it where the message says, then run this cell again.")
ready = pooled_path.exists() and (OUT_DIR / "panel_v2.csv").exists()
print("\nResults ready:", "yes" if ready else "not yet; the cells below wait until they are")


The rebuild command prints the step it is on, the rows it wrote and the seconds it took. A step with all its output
files already there says "skipped". If a file must be downloaded by hand, `fetch_public` prints the page, the exact
file and where to save it; the rebuild stops and waits for you. Start it again after you save the file.

## The real table

`OUT_DIR/panel_v2.csv` is the table that the models read: one row per annual report (10-K) of one company. `t` is the
snapshot date, `label` is 1 if S&P downgraded the company within 12 months, and `has_prices` is 1 when the stock-market
inputs were found. The next cell reads it and shows its size.

In [ ]:
if not ready:
    print("No real results yet: run the rebuild cell above first.")
else:
    panel = pd.read_csv(OUT_DIR / "panel_v2.csv")
    panel["t"] = pd.to_datetime(panel["t"])
    print(panel.shape[0], "rows,", panel.shape[1], "columns,", panel.cik.nunique(), "companies")
    show = [c for c in ["cik", "t", "label", "rating", "notch", "has_prices"] if c in panel]
    display(panel[show].head())


## Reports and downgrades by year

The table is easiest to read year by year. `reports` counts the annual reports whose snapshot date falls in that year,
`downgrades` counts the ones followed by an S&P downgrade, and `share` is the downgrade rate.

In [ ]:
if not ready:
    print("No real results yet: run the rebuild cell above first.")
else:
    panel["year"] = panel["t"].dt.year
    by_year = panel.groupby("year").label.agg(reports="size", downgrades="sum")
    by_year["share"] = (by_year.downgrades / by_year.reports).round(3)
    display(by_year)
    print("All years:", len(panel), "reports,", int(panel.label.sum()), "downgrades,", f"{panel.label.mean():.1%}")


Read the last line first: it is the overall chance that a report is followed by a downgrade. A model that always gives
that same chance is the baseline; the models have to beat it by ranking companies better.

## The pooled table

`OUT_DIR/run/pooled.csv` puts all test years together, one line per model and flag rule. The rule named
`calibrated_cut` is the lecture's flag line: the cut is set from an earlier year's scores, not from the test year
itself. `roc_auc` and `pr_auc` measure ranking quality (higher is better). `flagged`, `TP`, `FP` and `FN` count the
flag-line outcomes.

In [ ]:
if not ready:
    print("No real results yet: run the rebuild cell above first.")
else:
    pooled = pd.read_csv(pooled_path)
    show = pooled[pooled["rule"] == "calibrated_cut"] if "rule" in pooled else pooled
    columns = [c for c in ["model", "rule", "n", "downgrades", "roc_auc", "pr_auc", "flagged",
                           "TP", "FP", "FN", "miss_rate", "false_alarm_rate"] if c in show]
    display(show[columns])
    print("Models in the table:", ", ".join(show["model"].astype(str)))


## The flag-line counts, year by year

`OUT_DIR/run/per_year.csv` shows the same counts for each test year. `flagged` is the number of reports above the
earlier year's flag line, `TP` is a flagged report that was followed by a downgrade, `FP` is a flagged report with no
downgrade, and `FN` is a downgrade below the line (missed).

In [ ]:
if not ready:
    print("No real results yet: run the rebuild cell above first.")
else:
    per_year = pd.read_csv(OUT_DIR / "run" / "per_year.csv")
    if "rule" in per_year:
        per_year = per_year[per_year["rule"] == "calibrated_cut"]
    columns = [c for c in ["year", "model", "n", "downgrades", "flagged", "TP", "FP", "FN"] if c in per_year]
    display(per_year[columns])
    counts = [c for c in ["flagged", "TP", "FP", "FN"] if c in per_year]
    if counts:
        total = per_year[counts].sum()
        print("All test years:", ", ".join(f"{c} {int(total[c]):,}" for c in counts))


## What the numbers mean

- The free route and the WRDS route do **not** give the same table. Free Yahoo prices often have no file for a company
  that stopped trading. Those companies can be exactly the ones that were downgraded, so the free route can make the
  task look easier or harder than it is. The free route also covers only the companies whose SEC ticker was found.
- The WRDS route uses CRSP, the lecture's price data. Only that route can come close to the lecture's numbers; even
  then your result can differ because the rating files and the SEC files change over time.
- A flag is a reason to look at a company, not a reason to sell it. The `FN` count is the number of downgrades the flag
  line missed; no simple rule catches every one.

## Tidy up

The real data and the results are kept in `DATA_DIR` and `OUT_DIR`; this notebook does not delete them. To start fresh,
delete those two folders (or run the rebuild command with `--from 1`) when you are ready to download again.

## Check yourself

1. Why does the free route cover fewer companies than the WRDS route?
2. You run the free route and get a different ROC-AUC from the lecture. Is the code broken?
3. Why does the notebook check for `OUT_DIR/run/pooled.csv` before it runs the rebuild command?
4. In the per-year table, what is the difference between `FP` and `FN`?

<details>
<summary>Answers</summary>

1. Free Yahoo price files are missing for many companies that stopped trading (survivorship bias). The model needs
   stock-market inputs, so those report rows are left out. WRDS/CRSP keeps the delisted stocks.
2. Not necessarily. The free route replaces CRSP with Yahoo prices and covers about half the firms, so its table and
   therefore its scores differ. The WRDS route is the one that matches the lecture's data.
3. Rebuilding takes a long time. If the results are already there, the notebook reads them instead of downloading and
   running every step again. Use `--from 1` when you do want a fresh build.
4. `FP` (false alarm) is a flagged report that was **not** followed by a downgrade. `FN` (missed downgrade) is a
   downgrade that was **below** the flag line, so the model did not flag it.
</details>